# {mod}`ultralytics` 验证模式

验证模式：用于验证模型性能的训练后检查点。

验证是机器学习流水线中的关键步骤，它允许你评估训练模型的质量。Ultralytics YOLO11的验证模式提供了一套强大的工具和指标，用于评估你的目标检测模型的性能。

In [ ]:
import sys
_path = "/media/pc/data/lxw/ai/ultralytics" # ultralytics api 所在目录
sys.path.append(_path)
from pathlib import Path
from ultralytics import settings
temp_dir = Path("../.temp") # 设置缓存目录
temp_dir.mkdir(exist_ok=True, parents=True)
# 更新项目配置
settings.update({'weights_dir': f'{temp_dir}/weights'})

验证在 COCO8 数据集上训练的 YOLOv11n 模型的准确性。由于模型保留了其训练数据和参数作为模型属性，因此不需要额外的参数。

```python
from ultralytics import YOLO
model = YOLO("path/to/best.pt")  # load a custom model
# 验证模型
metrics = model.val()  # 无需额外参数，数据集及设置已被记忆。
```

直接使用官方模型：

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")  # 加载官方模型
metrics = model.val(data="coco8.yaml")
print(metrics.box.map)  # map50-95
print(metrics.box.map50)  # map50 （IoU 阈值为 0.5 时的平均精度平均值）
print(metrics.box.map75)  # map75 （在 IoU 临界值为 0.75 时的平均平均精度）
metrics.box.maps  # mAP50-95（从 0.5 到 0.95 的多个 IoU 阈值的平均精度平均值）

Ultralytics 8.3.28 🚀 Python-3.12.2 torch-2.5.0 CUDA:0 (NVIDIA GeForce RTX 3090, 24250MiB)
YOLO11n summary (fused): 238 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs


val: Scanning /media/pc/data/lxw/datasets/coco8/labels/val.cache... 4 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4/4 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  8.80it/s]


                   all          4         17       0.57       0.85      0.847      0.632
                person          3         10      0.557        0.6      0.585      0.272
                   dog          1          1      0.548          1      0.995      0.697
                 horse          1          2       0.53          1      0.995      0.674
              elephant          1          2       0.37        0.5      0.516      0.257
              umbrella          1          1      0.568          1      0.995      0.995
          potted plant          1          1      0.844          1      0.995      0.895
Speed: 0.3ms preprocess, 8.8ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to runs/detect/val5
0.6316492891584533
0.8469102486563518
0.6918055555555555


array([    0.27239,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,      0.6965,     0.67398,     0.63165,     0.63165,     0.25652,     0.63165,     0.63165,     0.63165,
           0.63165,       0.995,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,
           0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,      0.8955,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,     0.63165,
           0.63165,     0.63165,     0.6316

## YOLO 模型验证的参数

在验证YOLO模型时，可以对多个参数进行微调以优化评估过程。这些参数控制诸如输入图像大小、批处理和性能阈值等方面。下面详细分析每个参数，帮助您有效地自定义验证设置。

| Argument      | Type    | Default | Description                                                                                                                                                                                                                           |
| ------------- | ------- | ------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `data`        | `str`   | `None`  | Specifies the path to the dataset configuration file (e.g., `coco8.yaml`). This file includes paths to [validation data](https://www.ultralytics.com/glossary/validation-data), class names, and number of classes.                   |
| `imgsz`       | `int`   | `640`   | Defines the size of input images. All images are resized to this dimension before processing.                                                                                                                                         |
| `batch`       | `int`   | `16`    | Sets the number of images per batch. Use `-1` for AutoBatch, which automatically adjusts based on GPU memory availability.                                                                                                            |
| `save_json`   | `bool`  | `False` | If `True`, saves the results to a JSON file for further analysis or integration with other tools.                                                                                                                                     |
| `save_hybrid` | `bool`  | `False` | If `True`, saves a hybrid version of labels that combines original annotations with additional model predictions.                                                                                                                     |
| `conf`        | `float` | `0.001` | Sets the minimum confidence threshold for detections. Detections with confidence below this threshold are discarded.                                                                                                                  |
| `iou`         | `float` | `0.6`   | Sets the [Intersection Over Union](https://www.ultralytics.com/glossary/intersection-over-union-iou) (IoU) threshold for Non-Maximum Suppression (NMS). Helps in reducing duplicate detections.                                       |
| `max_det`     | `int`   | `300`   | Limits the maximum number of detections per image. Useful in dense scenes to prevent excessive detections.                                                                                                                            |
| `half`        | `bool`  | `True`  | Enables half-[precision](https://www.ultralytics.com/glossary/precision) (FP16) computation, reducing memory usage and potentially increasing speed with minimal impact on [accuracy](https://www.ultralytics.com/glossary/accuracy). |
| `device`      | `str`   | `None`  | Specifies the device for validation (`cpu`, `cuda:0`, etc.). Allows flexibility in utilizing CPU or GPU resources.                                                                                                                    |
| `dnn`         | `bool`  | `False` | If `True`, uses the [OpenCV](https://www.ultralytics.com/glossary/opencv) DNN module for ONNX model inference, offering an alternative to [PyTorch](https://www.ultralytics.com/glossary/pytorch) inference methods.                  |
| `plots`       | `bool`  | `False` | When set to `True`, generates and saves plots of predictions versus ground truth for visual evaluation of the model's performance.                                                                                                    |
| `rect`        | `bool`  | `True`  | If `True`, uses rectangular inference for batching, reducing padding and potentially increasing speed and efficiency.                                                                                                                 |
| `split`       | `str`   | `val`   | Determines the dataset split to use for validation (`val`, `test`, or `train`). Allows flexibility in choosing the data segment for performance evaluation.                                                                           |
| `project`     | `str`   | `None`  | Name of the project directory where validation outputs are saved.                                                                                                                                                                     |
| `name`        | `str`   | `None`  | Name of the validation run. Used for creating a subdirectory within the project folder, where valdiation logs and outputs are stored.                                                                                                 |


这些设置中的每一个都在验证过程中起着至关重要的作用，可以对YOLO 模型进行可定制的高效评估。根据您的具体需求和资源调整这些参数，有助于实现准确性和性能之间的最佳平衡。

## 带参数的验证示例
下面的示例展示了YOLO 模型验证自定义参数

In [9]:
from ultralytics import YOLO

# 加载模型
model = YOLO("yolo11n.pt")

# 自定义验证参数
validation_results = model.val(data="coco8.yaml", imgsz=640, batch=16, conf=0.25, iou=0.6, device="0")

Ultralytics 8.3.28 🚀 Python-3.12.2 torch-2.5.0 CUDA:0 (NVIDIA GeForce RTX 3090, 24250MiB)
YOLO11n summary (fused): 238 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs


val: Scanning /media/pc/data/lxw/datasets/coco8/labels/val.cache... 4 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4/4 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  4.57it/s]


                   all          4         17      0.694       0.65      0.698       0.51
                person          3         10      0.667        0.4      0.578      0.347
                   dog          1          1          1          1      0.995      0.697
                 horse          1          2          1          1      0.995       0.71
              elephant          1          2        0.5        0.5      0.622      0.311
              umbrella          1          1          1          1      0.995      0.995
          potted plant          1          1          0          0          0          0
Speed: 0.2ms preprocess, 17.9ms inference, 0.0ms loss, 5.0ms postprocess per image
Results saved to runs/detect/val6
